# Pertemuan 03 — Siklus Hidup Machine Learning dengan Scikit-Learn

Pada modul ini kita akan mempelajari **siklus hidup machine learning** secara praktik langsung menggunakan `scikit-learn`.
Kita akan melewati setiap tahap dalam siklus tersebut sambil mengenalkan perkakas dan teknik standar yang dipakai di dunia nyata.

Siklus hidup machine learning terdiri atas empat tahap:

| Tahap | Nama | Pertanyaan intinya |
|---|---|---|
| **L** | Permasalahan Pembelajaran | Apa yang ingin diprediksi, bagaimana menilai keberhasilan, dan data apa yang tersedia? |
| **M** | Perancangan Model | Fitur apa yang dipakai, dan keluarga model mana yang dipilih? |
| **O** | Optimisasi | Bagaimana model dilatih, dan bagaimana hyperparameter disetel? |
| **P** | Prediksi & Evaluasi | Seberapa baik model bekerja pada data yang belum pernah dilihat? |

<div style="text-align: center;">
<img src="https://eecs189.org/fa25/resources/assets/lectures/lec03/images/ml_lifecycle.png" alt="Siklus hidup machine learning" width="600"/>
</div>

> **Cara memakai modul ini.** Jalankan sel kode satu per satu dari atas ke bawah secara berurutan. Banyak sel bergantung pada hasil sel sebelumnya, sehingga melompati satu sel akan menyebabkan error. Setiap bagian diawali penjelasan singkat dan diakhiri pertanyaan untuk direnungkan.

**Pustaka yang dipakai:** `numpy` (komputasi numerik), `pandas` (data tabular), `plotly.express` (visualisasi interaktif), `scikit-learn` (machine learning), dan `torchvision` (untuk mengunduh dataset).


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

---
## 1. Permasalahan Pembelajaran (The Learning Problem)

Bayangkan kita sedang meluncurkan sebuah situs jual-beli fashion baru. Pengguna mengunggah foto pakaian yang ingin mereka tukarkan, dan kita ingin membantu mereka mengenali jenis pakaian pada foto tersebut secara otomatis. Kita sudah memiliki data latih berupa foto pakaian beserta labelnya (misalnya "dress", "shirt", "pants").

Sebelum menulis satu baris kode model pun, jawab dulu tiga pertanyaan berikut.

**Data apa yang kita miliki?**
* Contoh-contoh data latih yang sudah berlabel (*labeled training examples*).

**Apa yang ingin kita prediksi?**
* Label kategori pakaian pada gambar. Nantinya kita mungkin juga ingin memprediksi hal lain, misalnya harga atau ukuran.

**Bagaimana kita menilai keberhasilan?**
* Kemungkinan besar kita mengukur **akurasi** prediksi.
* Pada akhirnya kita mungkin ingin meningkatkan akurasi pada kelas-kelas tertentu yang bernilai tinggi secara bisnis.

> **Catatan.** Ketiga jawaban di atas adalah tahap **L (Learning Problem)**. Kesalahan paling mahal dalam proyek ML biasanya bukan salah memilih algoritma, melainkan salah merumuskan tiga pertanyaan ini.


### 1.1 Melihat Datanya Terlebih Dahulu

Langkah penting yang sering dilewatkan dalam proyek machine learning adalah **memahami datanya**. Ini mencakup menjelajahi dataset, memvisualkan isinya, dan menangkap gambaran mengenai struktur serta karakteristiknya.

Kita akan memakai dataset **Fashion-MNIST**, sebuah dataset klasik berisi gambar pakaian berukuran 28×28 piksel dalam skala abu-abu (*grayscale*).

> [Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms.](https://arxiv.org/abs/1708.07747) Han Xiao, Kashif Rasul, Roland Vollgraf.
> https://github.com/zalandoresearch/fashion-mnist

Dataset ini merupakan alternatif dari dataset MNIST yang lebih klasik lagi, yaitu kumpulan gambar angka tulisan tangan.


Blok kode berikut akan mengunduh dataset Fashion-MNIST dan memuatnya ke dalam memori. Proses unduhan hanya terjadi sekali; setelahnya data dibaca dari folder `data`.


In [ ]:
# Mengunduh dan memuat dataset
import torchvision
data = torchvision.datasets.FashionMNIST(root='data', train=True, download=True)

# Mengubah data menjadi array numpy
images = data.data.numpy().astype(float)
targets = data.targets.numpy() # label kelas dalam bentuk angka
class_dict = {i:class_name for i,class_name in enumerate(data.classes)}
labels = np.array([class_dict[t] for t in targets]) # label kelas dalam bentuk teks
n = len(images)

print("Dataset FashionMNIST dimuat dengan {} sampel.".format(n))
print("Kelas: {}".format(class_dict))
print("Ukuran gambar: {}".format(images[0].shape))
print("Tipe data gambar: {}".format(images[0].dtype))
print("Gambar ke-0:\n", images[0])

#### Memahami Fitur Mentahnya (Gambar)

**Berapa banyak data yang kita miliki?**

In [ ]:
images.shape

Gambar tersimpan dalam sebuah tensor berukuran 60000 × 28 × 28. Artinya kita punya 60.000 gambar, masing-masing selebar 28 piksel dan setinggi 28 piksel. Setiap piksel diwakili oleh satu angka.

**Nilai apa saja yang mungkin muncul pada piksel-piksel itu?** Mari kita lihat sebarannya.


In [ ]:
counts, bins =  np.histogram(images, bins=255)
fig_pixels = px.bar(x=bins[1:], y=counts,  title="Sebaran nilai piksel",
       log_y=True, labels={"x":"Nilai piksel", "y":"Jumlah"})
fig_pixels

Dari histogram di atas terlihat nilai piksel berkisar antara 0 sampai 255, dan sebagian besar bernilai 0 (latar belakang hitam). Perhatikan sumbu-y memakai skala logaritma agar nilai yang jarang muncul tetap terlihat.

Kemampuan memvisualkan data adalah keterampilan penting. Di sini kita memakai Plotly Express untuk menampilkan satu gambar. Kita memakai peta warna `'gray_r'`, yaitu skala abu-abu terbalik, sehingga nilai 0 tampak putih dan nilai tinggi tampak hitam.


In [ ]:
px.imshow(images[0], color_continuous_scale='gray_r')

Potongan kode berikut menampilkan beberapa gambar sekaligus dalam bentuk kisi. Anda tidak dituntut memahami detail kodenya, tetapi berguna untuk mengetahui cara menampilkan banyak gambar di Python.


In [ ]:
def show_images(images, max_images=40, ncols=5, labels = None):
    """Menampilkan sebagian gambar dari dataset dalam bentuk kisi.
    Argumen:
        images (np.ndarray): array gambar yang akan ditampilkan [gambar, baris, kolom].
        max_images (int): jumlah maksimum gambar yang ditampilkan.
        ncols (int): jumlah kolom pada kisi.
        labels (np.ndarray, opsional): label tiap gambar, dipakai sebagai judul.
    Mengembalikan:
        plotly.graph_objects.Figure: objek gambar Plotly.
    """
    n = min(images.shape[0], max_images) # banyaknya gambar yang ditampilkan
    px_height = 220 # tinggi tiap gambar dalam piksel
    fig = px.imshow(images[:n, :, :], color_continuous_scale='gray_r',
                    facet_col = 0, facet_col_wrap=ncols,
                    height = px_height * int(np.ceil(n/ncols)))
    fig.update_layout(coloraxis_showscale=False)
    if labels is not None:
        # Ambil nomor facet lalu ganti dengan labelnya.
        fig.for_each_annotation(lambda a: a.update(text=labels[int(a.text.split("=")[-1])]))
    return fig

In [ ]:
show_images(images, 20, labels=labels)

Sekarang mari lihat beberapa contoh dari setiap kelas. Di sini kita memakai pandas untuk mengelompokkan gambar berdasarkan labelnya, lalu mengambil 2 contoh acak dari tiap kelas.

Perhatikan pola `groupby` → `sample` → ambil indeksnya. Pola ini sangat sering dipakai untuk mengambil sampel yang mewakili seluruh kelas, bukan hanya yang kebetulan ada di baris-baris awal.


In [ ]:
idx = (
    pd.DataFrame({"labels": labels})
      .groupby("labels", as_index=False)
      .sample(2)
      .index
      .to_numpy())
show_images(images[idx,:,:], labels=labels[idx])

#### Memahami Labelnya

Sekarang kita periksa labelnya. Beberapa pertanyaan yang perlu dijawab:

* Apakah labelnya bersifat diskret (kategori) atau kontinu (angka)?
* Bagaimana sebarannya?
* Adakah nilai yang hilang atau salah?

Pada Fashion-MNIST, setiap gambar diberi label satu jenis pakaian. Seluruhnya ada 10 kelas.

Memahami **sebaran label** itu penting karena dapat mengungkap masalah seperti *class imbalance*, yaitu ketika sebagian kelas memiliki jauh lebih banyak contoh daripada kelas lain.


In [ ]:
labels

Labelnya berupa teks (bersifat diskret), bukan angka kontinu. Ini menegaskan bahwa persoalan kita adalah **klasifikasi**, bukan regresi.

**Bagaimana sebaran labelnya?**

In [ ]:
px.histogram(labels, title="Sebaran label")

Ternyata proporsi setiap jenis pakaian seimbang, masing-masing sekitar 6.000 gambar. Tidak ada nilai yang hilang karena semua label termasuk salah satu dari 10 kelas (tidak ada label kosong).

> **Perhatian.** Sebagian besar dataset dunia nyata tidak serapi dan seseimbang ini. Yang umum dijumpai justru sebaran *long tail*: beberapa kelas sangat sering muncul, sementara banyak kelas lain sangat jarang. Pada kasus seperti itu, akurasi saja menjadi metrik yang menyesatkan.


### 1.2 Menyimpulkan Jenis Permasalahannya

Setelah memeriksa data, kita tahu bahwa yang kita miliki adalah kumpulan besar pasangan **fitur** dan **label kategorikal** (10 kelas).

Karena itu:

* Ini adalah persoalan **supervised learning**, sebab tujuannya adalah mempelajari pemetaan dari fitur masukan (gambar) ke label keluaran (kategori) berdasarkan contoh berlabel.
* Karena labelnya diskret, ini adalah persoalan **klasifikasi** — lebih tepatnya **klasifikasi multi-kelas** dengan 10 kelas.
* Karena fitur masukannya berupa gambar, ini juga termasuk persoalan **computer vision**. Artinya, pada tahap perancangan model nanti kita perlu mempertimbangkan teknik yang memang dirancang untuk data citra.


---
*Kembali ke slide.*

---


### 1.3 Pembagian Data: Train — Validation — Test

Kita akan membagi dataset menjadi tiga bagian:

| Bagian | Kegunaan | Boleh dipakai berapa kali |
|---|---|---|
| **Training** | Melatih model (mencari nilai parameter) | Sesering mungkin |
| **Validation** | Menyetel hyperparameter dan membandingkan rancangan model | Berkali-kali |
| **Test** | Menilai kinerja akhir pada data yang benar-benar baru | **Sekali saja, di akhir** |

Secara teknis Fashion-MNIST sudah menyediakan data uji terpisah, tetapi di sini kita tetap mendemonstrasikan cara membagi data secara umum karena keterampilan ini akan selalu Anda butuhkan.

> **Mengapa perlu tiga bagian?** Kalau data uji dipakai untuk menyetel model, model itu secara tidak langsung sudah "melihat" data uji. Nilainya berhenti mengukur kemampuan generalisasi dan berubah menjadi angka yang terlalu optimistis.


In [ ]:
# memakai sklearn untuk membuat pembagian data latih dan uji
from sklearn.model_selection import train_test_split

In [ ]:
# Membuat pembagian data latih dan data uji
images_tr, images_te, labels_tr, labels_te = train_test_split(
    images, labels, test_size=0.2, random_state=42)

# Memisahkan data validasi dari data latih
images_tr, images_val, labels_tr, labels_val = train_test_split(
    images_tr, labels_tr, test_size=0.2, random_state=42)

print("Ukuran data latih   :", images_tr.shape)
print("Ukuran data validasi:", images_val.shape)
print("Ukuran data uji     :", images_te.shape)

Perhatikan bahwa pembagian dilakukan **dua kali**: pertama memisahkan data uji, lalu memisahkan data validasi dari sisa data latih. Hasil akhirnya kira-kira 64% latih, 16% validasi, dan 20% uji.

---
*Kembali ke slide.*

---


---
## 2. Perancangan Model (Model Design)

Dataset sudah dimuat pada bagian sebelumnya. Sekarang kita akan mengolah datanya agar siap dipakai melatih model klasifikasi.


### 2.1 Rekayasa Fitur (Feature Engineering)

**Rekayasa fitur** adalah proses mengubah fitur mentah menjadi representasi yang dapat dipakai secara efektif oleh teknik machine learning. Dalam praktiknya, ini hampir selalu berarti mengubah data menjadi **representasi vektor angka**.


#### Featurisasi Gambar secara Sederhana

Pada contoh ini kita bekerja dengan gambar skala abu-abu beresolusi rendah. Kita memakai pendekatan featurisasi paling sederhana, yaitu **meratakan (flatten)** gambar: mengubah setiap gambar 28×28 piksel menjadi satu vektor berdimensi 784 (karena 28 × 28 = 784).

> **Apa yang hilang dari cara ini?** Informasi mengenai posisi piksel yang bertetangga. Setelah diratakan, model tidak lagi tahu bahwa piksel ke-1 dan ke-29 sebenarnya bersebelahan secara vertikal. Inilah salah satu alasan jaringan konvolusi (CNN) bekerja jauh lebih baik untuk citra — topik yang akan dibahas pada pertemuan berikutnya.


In [ ]:
images_tr.shape

In [ ]:
# Mengubah tiap gambar 28x28 menjadi satu vektor berdimensi 784
def flatten(images):
    return images.reshape(images.shape[0], -1)

In [ ]:
X_tr = flatten(images_tr)

In [ ]:
X_tr.shape

#### Standarisasi Fitur

Ingat bahwa intensitas piksel berkisar dari 0 sampai 255:


In [ ]:
fig_pixels

Mari kita standarkan intensitas piksel agar berrata-rata nol dan bervariansi satu.

Di sini kita memakai `StandardScaler` dari sklearn. Perhatikan polanya yang berlaku untuk hampir semua alat pengubah data di sklearn:

1. **Konstruktor** — membuat objeknya, misalnya `StandardScaler()`
2. **`fit`** — mempelajari parameter transformasi (di sini: rata-rata dan variansi tiap piksel)
3. **`transform`** — menerapkan transformasi itu pada data

> **Aturan penting.** `fit` hanya boleh dilakukan pada **data latih**. Bila kita melakukan `fit` pada seluruh data, informasi dari data uji ikut bocor ke dalam transformasi (*data leakage*) dan hasil evaluasi menjadi terlalu bagus.


In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Membuat objek StandardScaler
image_scaler = StandardScaler()

# 2. Mempelajari rata-rata dan variansi dari data latih
image_scaler.fit(flatten(images_tr))

**Apa yang dapat kita simpulkan dari citra rata-rata dan citra variansi mengenai dataset ini?**

Perhatikan bahwa bagian tengah gambar memiliki rata-rata dan variansi yang tinggi (di situlah objek pakaian berada), sedangkan bagian pinggir hampir selalu bernilai nol. Piksel di pinggir nyaris tidak membawa informasi.


In [ ]:
display(px.imshow(image_scaler.mean_.reshape(28,28),
                  color_continuous_scale='gray_r', title="Citra rata-rata"))
display(px.imshow(image_scaler.var_.reshape(28,28),
                  color_continuous_scale='gray_r', title="Citra variansi"))

Mari kita buat satu fungsi featurisasi umum yang dapat dipakai ulang untuk data latih, validasi, maupun uji.

Perhatikan bahwa fungsi ini memakai `image_scaler` yang sudah di-`fit` pada data latih. Dengan begitu, data validasi dan uji diproses memakai statistik dari data latih — persis seperti yang seharusnya.


In [ ]:
# Fungsi featurisasi: ratakan gambar, lalu terapkan skala dari data latih
def featurizer(images):
    flattened = flatten(images)
    return image_scaler.transform(flattened)

X_tr = featurizer(images_tr)

Gambar hasil standarisasi terlihat mirip dengan aslinya, tetapi nilainya sudah diubah agar berrata-rata nol dan bervariansi satu. Ini umumnya membantu meningkatkan kinerja dan mempercepat pelatihan model.

In [ ]:
show_images(X_tr.reshape(images_tr.shape), max_images=10, labels=labels_tr)

#### One-Hot Encoding

Dataset gambar ini tidak membutuhkan one-hot encoding, tetapi teknik ini sangat penting untuk data **kategorikal**. Kita demonstrasikan sebentar dengan dataset kecil buatan.

One-hot encoding mengubah satu kolom kategori menjadi beberapa kolom biner — satu kolom untuk setiap nilai yang mungkin. Dengan begitu model tidak salah mengira bahwa kategori punya urutan atau nilai yang lebih besar.


In [ ]:
df = pd.DataFrame({"warna": ["merah", "hijau", "merah", "biru", "biru", "kuning", ""]})
df

In [ ]:
from sklearn.preprocessing import OneHotEncoder
# 1. Membuat objek OneHotEncoder
ohe = OneHotEncoder()
# 2. Mempelajari daftar kategorinya
ohe.fit(df[["warna"]])

In [ ]:
ohe.categories_

In [ ]:
ohe.transform(df[["warna"]]).toarray()
ohe.categories_

#### Bag of Words

Representasi bag-of-words juga tidak kita perlukan untuk dataset gambar ini, tetapi mari kita lihat sekilas dengan contoh data teks.

Bag-of-words mengubah setiap kata dalam kosakata menjadi satu kolom, lalu mengisinya dengan jumlah kemunculan kata itu pada masing-masing dokumen. **Urutan kata diabaikan** — itulah keterbatasan utamanya.


In [ ]:
df['teks'] = [
    "Merah adalah sebuah warna.",
    "Hijau untuk makanan hijau.",
    "Merah mengingatkan saya pada makanan merah.",
    "Biru adalah warna kesukaan saya!",
    "Biru untuk kampus!",
    "Kuning juga untuk kampus!",
    "Saya lupa menulis sesuatu."
]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Membuat objek CountVectorizer
vectorizer = CountVectorizer()

# 2. Membangun kosakata dari teks
vectorizer.fit(df["teks"])


In [ ]:
pd.DataFrame(vectorizer.transform(df["teks"]).toarray(),
             columns=vectorizer.get_feature_names_out())

Perhatikan hasilnya: setiap baris menjadi vektor panjang yang hampir seluruhnya bernilai nol. Representasi seperti ini disebut *sparse* (jarang).

---
*Kembali ke slide.*

---


---
## 3. Pemodelan dan Optimisasi

Pada bagian ini kita menjalani proses pemodelan, dengan fokus mengembangkan sebuah model klasifikasi.


### 3.1 Melatih (Fitting) Sebuah Classifier

Kita mulai dari classifier yang paling dasar, yaitu **regresi logistik** (*logistic regression*), untuk mendemonstrasikan alur kerja klasifikasi.

Regresi logistik adalah **model linear** yang lazim dipakai untuk klasifikasi biner maupun multi-kelas. Model ini juga menjadi titik awal yang baik untuk memahami model deep learning yang lebih rumit nanti.

Di sini kita memakai `sklearn` untuk melatih model regresi logistik pada data latih. Kelas `LogisticRegression` dari `sklearn.linear_model` dipakai untuk membuat objek modelnya.

Metode `fit` dipanggil pada objek model dengan memberikan data latih beserta labelnya. Proses ini melatih model untuk mempelajari hubungan antara fitur masukan (gambar yang sudah diratakan) dan label targetnya (kategori pakaian). Di scikit-learn, metode `fit` selalu dipakai untuk melatih model apa pun.

> **Sabar ya.** Sel di bawah ini butuh waktu cukup lama untuk dijalankan karena datanya besar (38.400 gambar × 784 fitur).


In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression()
lr_model.fit(X=X_tr, y=labels_tr)

Perhatikan bahwa kita mendapat peringatan seperti berikut:

```plaintext
lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
```

Peringatan ini menandakan bahwa **algoritma optimisasi** yang dipakai regresi logistik belum mencapai solusi (belum *konvergen*) dalam jumlah iterasi bawaan. Hal ini bisa terjadi bila modelnya rumit atau datanya tidak berskala baik.

Kita dapat mengganti algoritma optimisasi yang dipakai. Algoritma bawaannya adalah `lbfgs`, sebuah metode kuasi-Newton. Pilihan lainnya antara lain `newton-cg`, `sag`, dan `saga`. Masing-masing punya kelebihan dan kekurangan, dan pilihan algoritma memengaruhi kecepatan konvergensi serta kinerja akhir model.

Pada mata kuliah ini kita akan menjelajahi variasi *stochastic gradient descent* seperti `saga`. Mari coba `saga` sebagai ganti `lbfgs` dan lihat apakah konvergensinya lebih cepat. Kita juga menaikkan `tol` (batas toleransi) agar pelatihan berhenti lebih awal dan kita tidak perlu menunggu terlalu lama.

> **Ingat.** `solver` dan `tol` adalah **hyperparameter**: kita yang menentukannya sebelum pelatihan dimulai, bukan model yang mempelajarinya.


In [ ]:
lr_model = LogisticRegression(tol=0.05, solver='saga', random_state=42)
lr_model.fit(X=X_tr, y=labels_tr)

### 3.2 Parameter

**Parameter** adalah variabel internal yang **dipelajari model sendiri** selama proses pelatihan. Pada regresi logistik, parameternya adalah bobot (*weights*) yang diberikan kepada setiap fitur beserta intersepnya. Bobot-bobot ini disesuaikan selama pelatihan untuk meminimalkan fungsi kerugian (*loss function*), yaitu ukuran seberapa jauh prediksi model meleset dari label sebenarnya.

Mari kita lihat bentuk dan isi parameter model yang baru saja dilatih.


In [ ]:
print("model.coef_.shape:", lr_model.coef_.shape)
print("model.intercept_.shape:", lr_model.intercept_.shape)
print(lr_model.coef_)
print(lr_model.intercept_)

Koefisien tersebut dapat kita visualkan. Karena setiap kelas memiliki 784 bobot — satu untuk tiap piksel — kita bisa menyusunnya kembali menjadi gambar 28×28.

Hasilnya membantu memahami piksel mana yang paling menentukan untuk setiap kelas: area gelap berarti bobot besar yang mendukung kelas itu, area terang berarti bobot yang menentangnya. Anda belum perlu memahami detail matematisnya sekarang.


In [ ]:
coeffs = lr_model.coef_
show_images(coeffs.reshape(10, 28, 28), labels=lr_model.classes_)

#### Jaringan Saraf (Neural Networks)

Jaringan saraf sering menjadi pilihan utama untuk tugas klasifikasi citra. Model ini mampu mempelajari pola yang rumit dan biasanya mengungguli model sederhana seperti regresi logistik. Namun ia juga menuntut arsitektur yang tepat, data latih yang jauh lebih banyak, serta sumber daya komputasi yang besar.

Di sini kita mencoba jaringan saraf sederhana dengan **dua lapisan tersembunyi** berukuran 100 dan 50 neuron.

> **Sabar lagi ya.** Sel ini juga butuh waktu beberapa menit. Perhatikan bahwa `hidden_layer_sizes`, `max_iter`, dan `tol` semuanya adalah hyperparameter.


In [ ]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    max_iter=100, tol=1e-3, random_state=42)

mlp.fit(X=X_tr, y=labels_tr)

### 3.3 Hyperparameter

**Hyperparameter** adalah argumen yang kita tetapkan **sebelum** proses pelatihan dimulai. Termasuk di dalamnya adalah pilihan algoritma optimisasi, laju pembelajaran, jumlah iterasi maksimum, dan kekuatan regularisasi. Hyperparameter biasanya disetel dengan teknik seperti validasi silang (*cross-validation*) untuk menemukan kombinasi terbaik bagi suatu dataset.

Agak membingungkan, hyperparameter ini sering disebut sekadar "parameter" dalam konteks pustaka seperti `sklearn`. Contohnya, kelas `LogisticRegression` memiliki hyperparameter `solver`, `C`, dan `max_iter` yang dapat diatur untuk memperbaiki kinerja model.

**Ringkasnya:**

| | Parameter | Hyperparameter |
|---|---|---|
| Siapa yang menentukan | Dipelajari model dari data | Ditetapkan manusia |
| Kapan | Selama pelatihan | Sebelum pelatihan |
| Contoh | `coef_`, `intercept_` | `C`, `solver`, `max_iter` |
| Disetel memakai | Algoritma optimisasi | Data validasi |

Menampilkan objek modelnya akan memperlihatkan seluruh hyperparameter yang sedang berlaku:


In [ ]:
lr_model

Untuk mengevaluasi model, kita memakai **data validasi**. Jangan lupa data validasi harus melewati fungsi featurisasi yang sama dengan data latih.

In [ ]:
X_val = featurizer(images_val)

Mari kita coba menyetel parameter regularisasi `C`. Agar prosesnya lebih menggambarkan konsepnya, kita bekerja dengan subhimpunan data latih yang lebih kecil (n = 1000). Ini membuat gejala *underfitting* dan *overfitting* jauh lebih kelihatan sekaligus mempercepat pelatihan.

> **Cara membaca `C`.** Pada sklearn, `C` adalah kebalikan dari kekuatan regularisasi. **`C` kecil berarti regularisasi kuat** (model lebih sederhana), sedangkan **`C` besar berarti regularisasi lemah** (model lebih bebas mengikuti data latih).


In [ ]:
# Ambil 1000 contoh pertama saja agar pelatihan cepat dan efek regularisasi terlihat jelas
n_small = 1000
X_tr_small = X_tr[:n_small,:]
labels_tr_small = labels_tr[:n_small]

Dalam praktiknya, kita perlu berhati-hati saat menyetel parameter regularisasi memakai sampel data yang kecil. Pada kasus ini, data yang lebih sedikit umumnya membutuhkan regularisasi yang lebih kuat untuk mencegah overfitting.

Sel berikut melatih 20 model dengan nilai `C` berbeda, lalu mencatat log-probabilitas rata-rata dan akurasinya pada data latih maupun data validasi. **Sel ini butuh waktu beberapa menit.**


In [ ]:
from sklearn.metrics import log_loss

C_vals = np.logspace(-5, 1, 20)

logprob_tr = []
logprob_val = []
acc_tr = []
acc_val = []

for C in C_vals:
    print("Mulai melatih dengan C =", C)
    model = LogisticRegression(tol=1e-3, random_state=42, C=C)
    model.fit(X=X_tr_small, y=labels_tr_small)

    # menghitung rata-rata log-probabilitas
    logprob_tr.append(-log_loss(labels_tr_small, model.predict_proba(X_tr_small), labels=model.classes_))
    logprob_val.append(-log_loss(labels_val, model.predict_proba(X_val), labels=model.classes_))

    # menghitung akurasi
    acc_tr.append(np.mean(model.predict(X_tr_small) == labels_tr_small))
    acc_val.append(np.mean(model.predict(X_val) == labels_val))


In [ ]:
df_logprob = pd.DataFrame({
    "C_val": C_vals,
    "Latih": logprob_tr, "Validasi": logprob_val,
}).set_index("C_val")

display(
    px.line(df_logprob,
        labels={"value": "Rata-rata Log Prob.", "C_val": "Parameter Regularisasi C", "variable": "Data"},
        title="Log-Probabilitas Regresi Logistik terhadap Parameter Regularisasi",
        markers=True,
        log_x=True,
        width=800, height=500)
)

df_acc = pd.DataFrame({
    "C_val": C_vals,
    "Latih": acc_tr, "Validasi": acc_val
}).set_index("C_val")

display(
    px.line(df_acc,
        labels={"value": "Akurasi", "C_val": "Parameter Regularisasi C", "variable": "Data"},
        title="Akurasi Regresi Logistik terhadap Parameter Regularisasi",
        markers=True,
        log_x=True,
        width=800, height=500
    )
)

**Cara membaca kedua grafik di atas:**

* Kurva **Train** hampir selalu naik terus seiring `C` membesar (regularisasi melemah) — model makin bebas mencocokkan data latih.
* Kurva **Validation** naik lebih dulu, mencapai puncak, lalu **menurun**. Titik puncak itulah kombinasi terbaik.
* Sebelah kiri puncak = **underfitting** (regularisasi terlalu kuat). Sebelah kanan puncak = **overfitting** (regularisasi terlalu lemah).

Pola inilah yang selalu kita cari setiap kali menyetel hyperparameter apa pun.

---
## 4. Mengevaluasi Model

Setelah model dilatih, kita dapat memakainya untuk membuat prediksi pada data baru. Metode `predict` pada model terlatih dipakai untuk menghasilkan prediksi berdasarkan fitur masukan.


Mari kembali ke model regresi logistik kita.

In [ ]:
lr_model.predict(X_tr[:10,:])

**Apakah Anda setuju dengan prediksinya?** Mari kita visualkan prediksi pada beberapa gambar.

In [ ]:
show_images(images_tr[:10,:].reshape(10, 28, 28),
            labels = lr_model.predict(X_tr[:10,:]))

Sekarang mari bandingkan dengan label yang sebenarnya. Judul setiap gambar berisi label asli, diikuti prediksi model dalam kurung.

In [ ]:
k = 10
tmp_labels = labels_tr[:k] + " (prediksi=" + lr_model.predict(X_tr[:k,:]) + ")"
show_images(images_tr[:k,:].reshape(k, 28, 28), labels=tmp_labels)

### 4.1 Memprediksi Probabilitas

Banyak model juga dapat memberikan probabilitas untuk setiap kelas melalui metode `predict_proba`. Ini berguna untuk memahami **seberapa yakin** model terhadap prediksinya.

Pada mata kuliah ini kita akan sering memakai kerangka probabilistik, yaitu menafsirkan keluaran model sebagai probabilitas tiap kelas.

> **Mengapa penting?** `predict` hanya menjawab "Sweater". `predict_proba` menjawab "Sweater 62%, Kaos 25%, Jaket 13%" — jawaban kedua jauh lebih berguna bila keputusannya berisiko, karena kita tahu model sebenarnya ragu.


In [ ]:
lr_model.predict_proba(X_tr[:5,:])

Mari kita visualkan probabilitas tersebut untuk gambar-gambar yang tadi diprediksi. Batang yang tinggi pada satu warna berarti model sangat yakin; batang yang terbagi rata ke beberapa warna berarti model ragu.

In [ ]:
k = 10
df = pd.DataFrame(lr_model.predict_proba(X_tr[:k,:]), columns=lr_model.classes_)
bars = px.bar(df, barmode='stack',orientation='v')
bars.update_layout(xaxis_tickmode='array', xaxis_tickvals=np.arange(k))
display(bars)
tmp_labels = labels_tr[:k] + " (prediksi=" + lr_model.predict(X_tr[:k,:]) + ") gbr: " + np.arange(k).astype(str)
show_images(images_tr[:k,:].reshape(k, 28, 28), labels=tmp_labels)

### 4.2 Metrik Akurasi dan Kinerja pada Data Uji

Setelah model dilatih, kita ingin mengevaluasinya. Ada banyak cara mengevaluasi model, dan cara terbaiknya bergantung pada jenis tugas serta datanya. Untuk klasifikasi, metrik yang lazim dipakai antara lain akurasi, *precision*, *recall*, dan *F1-score*. Kita mulai dari akurasi.

**Akurasi** adalah metrik paling sederhana: proporsi prediksi yang benar terhadap seluruh prediksi.


Mari mulai dengan menghitung akurasi model pada **data latih**.

In [ ]:
np.mean(lr_model.predict(X_tr) == labels_tr, axis=0)

Masalahnya, angka pada data latih bisa menyesatkan karena model mungkin sudah *overfitting* — bekerja baik pada data latih tetapi buruk pada data yang belum pernah dilihat. Secara intuitif, ini seperti berlatih dengan sekumpulan soal lalu berhasil menjawab soal-soal yang sama persis di ujian, tetapi kebingungan begitu soalnya diganti sedikit.

Untuk menilai kinerja model pada data yang belum pernah dilihat, kita evaluasi pada **data uji**. Ingat, data uji adalah bagian dataset yang sama sekali tidak dipakai selama pelatihan.


In [ ]:
X_te = featurizer(images_te)

In [ ]:
np.mean(lr_model.predict(X_te) == labels_te, axis=0)

In [ ]:
from sklearn.metrics import accuracy_score

train_acc = accuracy_score(labels_tr, lr_model.predict(X_tr))
val_acc = accuracy_score(labels_val, lr_model.predict(X_val))
test_acc = accuracy_score(labels_te, lr_model.predict(X_te))

print("Akurasi data latih   :", train_acc)
print("Akurasi data validasi:", val_acc)
print("Akurasi data uji     :", test_acc)

Akurasi pada data uji sedikit lebih rendah daripada data latih, dan itu memang wajar. Namun selisihnya tidak besar, yang menandakan model belum mengalami overfitting yang parah.

Kalau selisihnya besar (misalnya latih 99% tetapi uji 70%), itu tanda kuat overfitting.


**Apakah akurasi ini sudah bagus? Berapa akurasi yang dihasilkan oleh tebakan acak?**

Cara umum mengevaluasi model klasifikasi adalah membandingkan akurasinya terhadap sebuah **baseline**. Baseline paling sederhana adalah tebakan acak, yaitu memberi kelas secara acak pada setiap gambar.

**Berapa akurasi yang dihasilkan tebakan acak?**

Jawabannya bergantung pada seberapa sering setiap kelas muncul di data uji. Karena Fashion-MNIST seimbang dengan 10 kelas, tebakan acak akan menghasilkan akurasi sekitar 1/10 = 10%.


In [ ]:
np.random.seed(42)
print("Akurasi model      :", np.mean(lr_model.predict(X_val) == labels_val, axis=0))
print("Akurasi tebakan acak:",
      np.mean(np.random.choice(lr_model.classes_, size=len(labels_te)) == labels_te, axis=0))

**Apakah model kita kesulitan pada kelas tertentu?**

Sel berikut menampilkan sebaran label sebenarnya dari prediksi yang salah. Kelas dengan batang tertinggi adalah kelas yang paling sering disalahprediksi.


In [ ]:
isWrong = lr_model.predict(X_val) != labels_val
# membuat histogram frekuensi prediksi yang salah per kelas
fig = px.histogram(labels_val[isWrong], histnorm='percent')
fig.update_layout(xaxis_title="Label sebenarnya",
                  yaxis_title="Persentase prediksi salah")
fig.update_xaxes(categoryorder="total descending")

Untuk tugas klasifikasi, kita sering ingin melihat lebih dari sekadar akurasi. **Confusion matrix** membantu memvisualkan kinerja model pada tiap kelas: ia menunjukkan berapa banyak prediksi benar dan salah untuk setiap kelas.

**Cara membacanya:** baris menyatakan label sebenarnya, kolom menyatakan label yang diprediksi. Sel-sel di diagonal utama adalah prediksi yang benar. Sel di luar diagonal yang berwarna gelap menunjukkan pasangan kelas yang sering tertukar — misalnya kemeja dengan kaus, yang memang mirip bahkan bagi mata manusia.


In [ ]:
from sklearn.metrics import confusion_matrix

fig = px.imshow(
    confusion_matrix(labels_val, lr_model.predict(X_val)),
    color_continuous_scale='Blues'
    )
fig.update_layout(
        xaxis_title="Label prediksi",
        yaxis_title="Label sebenarnya",
        coloraxis_showscale=False,
        xaxis=dict(tickmode='array', tickvals=np.arange(len(lr_model.classes_)), ticktext=lr_model.classes_),
        yaxis=dict(tickmode='array', tickvals=np.arange(len(lr_model.classes_)), ticktext=lr_model.classes_)
    )


---
## 5. Penutup

Pada tugas nanti, Anda akan berkesempatan bekerja dengan data ini dan memakai scikit-learn lebih mendalam. Kami menyarankan Anda membaca dokumentasi dan tutorial di situs scikit-learn seiring berjalannya perkuliahan. Keduanya adalah sumber yang sangat baik untuk memahami berbagai fungsi dan kemampuan pustaka ini sekaligus konsep machine learning itu sendiri.

### Ringkasan alur yang baru saja kita lalui

| Tahap | Yang kita kerjakan | Perkakas |
|---|---|---|
| **L** Permasalahan | Merumuskan target, tujuan, dan data; memeriksa sebaran piksel dan label | `plotly.express`, `pandas` |
| **L** Pembagian data | Memisahkan train / validation / test | `train_test_split` |
| **M** Rekayasa fitur | Meratakan gambar dan menstandarkan piksel | `StandardScaler` |
| **M** Keluarga model | Regresi logistik dan jaringan saraf sederhana | `LogisticRegression`, `MLPClassifier` |
| **O** Optimisasi | Mengganti solver dan menyetel regularisasi `C` | argumen `solver`, `tol`, `C` |
| **P** Prediksi & evaluasi | Prediksi label dan probabilitas, akurasi, confusion matrix | `predict`, `predict_proba`, `confusion_matrix` |

### Latihan mandiri

1. Ganti nilai `random_state` pada `train_test_split` menjadi angka lain. Apakah akurasi uji berubah? Sebesar apa? Apa artinya bagi cara kita melaporkan hasil?
2. Latih model **tanpa** standarisasi (pakai `flatten(images_tr)` langsung). Bandingkan akurasi dan waktu pelatihannya.
3. Bandingkan akurasi validasi antara `lr_model` dan `mlp`. Model mana yang menang, dan berapa harga yang harus dibayar untuk kemenangan itu?
4. Dari confusion matrix, temukan dua kelas yang paling sering tertukar. Tampilkan beberapa gambarnya dan nilai sendiri: apakah Anda sebagai manusia juga akan keliru?
5. Hitung *precision*, *recall*, dan *F1-score* per kelas memakai `sklearn.metrics.classification_report`. Kelas mana yang paling sulit bagi model?

### Bacaan lanjutan

* Dokumentasi scikit-learn: https://scikit-learn.org/stable/
* Panduan transformasi data: https://scikit-learn.org/stable/data_transforms.html
* Galeri Plotly Express: https://plotly.com/python/plotly-express/
